# 🧪 Praktikum 01 – Entwicklungsumgebung & erste LLM-Interaktion
**Applied Generative AI – NLP | Sommersemester 2026**

> ⏱️ **Gesamtdauer: ~90 Minuten**  
> 🎯 **Lernziele:** Python-Umgebung verifizieren · Ollama installieren & bedienen · lokale LLM-Inferenz via REST-API und Python-Library · Systemparameter erkunden

---

## Vorbereitung – Was du brauchst
| Tool | Version | Installationsquelle |
|------|---------|---------------------|
| Python | 3.11+ | python.org |
| Ollama | aktuell | [ollama.com](https://ollama.com) |
| Jupyter Lab/Notebook | aktuell | `pip install jupyterlab` |

**Terminal-Befehl für eine lokale Vorab-Installation (optional):**
```bash
ollama pull qwen3.5:0.8b
ollama serve
```


In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

print("Runtime:", "Google Colab" if IN_COLAB else "Lokal/Jupyter")
print("\nOptional: Ollama-Konfiguration über Umgebungsvariablen")
print("  OLLAMA_BASE_URL (Default: http://127.0.0.1:11434)")
print("  LLM_MODEL (Default: qwen3.5:0.8b)")

# Optional before the setup cell:
# os.environ["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
# os.environ["LLM_MODEL"] = "qwen3.5:0.8b"

if IN_COLAB:
    print("\nColab-Modus: Ollama wird in der Setup-Zelle automatisch installiert und gestartet.")

## Studentische Checkliste (Start)

1. Setup-Zelle ausführen. In Colab werden Ollama und das Modell automatisch vorbereitet.
2. Optional vor dem Setup `OLLAMA_BASE_URL` und `LLM_MODEL` setzen.
3. Danach das Notebook einfach von oben nach unten durchlaufen.

## Teil 1 – Umgebung prüfen ⏱️ ~10 min

In [ ]:
import importlib
import os
import platform
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
AUTO_INSTALL_MISSING = True

REQUIRED = {
    "requests": "2.33.1",
    "ollama": "0.6.1",
    "rich": "13.9.4",
}


def run_install(specs):
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if in_venv:
        commands = [
            [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, *specs],
            ["uv", "pip", "install", "--python", sys.executable, *specs],
            [sys.executable, "-m", "pip", "install", *specs],
        ]
    else:
        commands = [
            [sys.executable, "-m", "uv", "pip", "install", "--system", *specs],
            ["uv", "pip", "install", "--system", *specs],
            [sys.executable, "-m", "pip", "install", "--break-system-packages", *specs],
        ]
    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Paketinstallation fehlgeschlagen: {last_error}")


missing = [name for name in REQUIRED if find_spec(name) is None]
if missing:
    print("Fehlende Pakete:", ", ".join(missing))
    if not AUTO_INSTALL_MISSING:
        raise RuntimeError("AUTO_INSTALL_MISSING ist False, aber Pakete fehlen.")
    specs = [f"{name}=={REQUIRED[name]}" for name in missing]
    run_install(specs)
    importlib.invalidate_caches()

missing_after = [name for name in REQUIRED if find_spec(name) is None]
if missing_after:
    raise RuntimeError(f"Diese Pakete fehlen weiterhin: {', '.join(missing_after)}")

import requests
from ollama import Client

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
DEFAULT_OPTIONS = {
    "num_ctx": 8192,
    "num_predict": 512,
    "temperature": 0.5,
}


def build_options(**overrides):
    options = DEFAULT_OPTIONS.copy()
    for key, value in overrides.items():
        if value is not None:
            options[key] = value
    return options


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


_OLLAMA_LOG_HANDLE = None
OLLAMA_SERVER_PROCESS = None


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama-Server unter {base_url} nicht erreichbar: {last_error}")


if is_local_host(OLLAMA_BASE_URL):
    if shutil.which("ollama") is None:
        if not IN_COLAB:
            raise RuntimeError(
                "Ollama ist lokal nicht installiert. Bitte Ollama installieren und die Setup-Zelle erneut ausführen."
            )
        subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

    try:
        wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
    except RuntimeError:
        log_path = "/tmp/ollama-notebook.log"
        _OLLAMA_LOG_HANDLE = open(log_path, "a", encoding="utf-8")
        OLLAMA_SERVER_PROCESS = subprocess.Popen(
            ["ollama", "serve"],
            stdout=_OLLAMA_LOG_HANDLE,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

    env = dict(os.environ)
    env["OLLAMA_HOST"] = OLLAMA_BASE_URL
    subprocess.check_call(["ollama", "pull", MODEL], env=env)

OLLAMA_CLIENT = Client(host=OLLAMA_BASE_URL)
model_payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
AVAILABLE_MODELS = [item.get("name", "") for item in model_payload.get("models", []) if isinstance(item, dict)]
if MODEL not in AVAILABLE_MODELS:
    raise RuntimeError(f"Modell '{MODEL}' wurde nicht gefunden.")

print("=" * 55)
print("  SYSTEMCHECK")
print("=" * 55)
print(f"Python      : {sys.version}")
print(f"Platform    : {platform.platform()}")
print(f"Runtime     : {'Google Colab' if IN_COLAB else 'Lokal/Jupyter'}")
for pkg in REQUIRED:
    print(f"  {'✅' if find_spec(pkg) else '❌'}  {pkg:<12}")
print(f"Ollama-Base-URL: {OLLAMA_BASE_URL}")
print(f"Modell         : {MODEL}")
print(f"Verfügbare Modelle: {', '.join(AVAILABLE_MODELS)}")

## Teil 2 – Ollama API direkt mit `requests` ⏱️ ~20 min

Ollama stellt unter `http://localhost:11434` eine REST-API bereit.  
Wir nutzen zunächst das rohe HTTP-Interface, um zu verstehen, wie LLM-Kommunikation *unter der Haube* funktioniert.

### 2.1 – Verfügbare Modelle abfragen

In [ ]:
import json
import requests

resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
resp.raise_for_status()
models = resp.json().get("models", [])

print(f"Lokale Modelle ({len(models)}):")
for m in models:
    size_gb = m.get("size", 0) / 1e9
    print(f"  • {m['name']:<30}  {size_gb:.1f} GB")

### 2.2 – Einfache Completion (non-streaming)

In [ ]:
payload = {
    "model": MODEL,
    "prompt": "Erkläre in einem Satz, was ein Transformer ist.",
    "stream": False,
    "think": False,
    "options": build_options(temperature=0.3)
}

resp = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload, timeout=120)
resp.raise_for_status()

data = resp.json()
print("Antwort:", data["response"])
print()
print(f"Tokens erzeugt   : {data.get('eval_count', '?')}")
print(f"Tokens/Sek       : {data.get('eval_count', 0) / max(data.get('eval_duration', 1), 1) * 1e9:.1f}")
print(f"Gesamtdauer      : {data.get('total_duration', 0) / 1e9:.2f}s")

### 2.3 – Streaming-Antwort in Echtzeit

In [ ]:
payload_stream = {
    "model": MODEL,
    "prompt": "Nenne die 5 wichtigsten Meilensteine in der KI-Geschichte.",
    "stream": True,
    "think": False,
    "options": build_options(temperature=0.1)
}

print("Streaming-Antwort von", MODEL, "...")
print("URL:", OLLAMA_BASE_URL)
print("-" * 50)

with requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload_stream, stream=True, timeout=180) as r:
    r.raise_for_status()
    for line in r.iter_lines():
        if line:
            chunk = json.loads(line)
            print(chunk.get("response", ""), end="", flush=True)
            if chunk.get("done"):
                break

print()
print("-" * 50)

## Teil 3 – Chat-Interface mit der `ollama` Python-Library ⏱️ ~20 min

Die `ollama`-Library ist eine höhere Abstraktion über die REST-API  
und unterstützt **Multi-Turn-Conversations** mit Rollenverwaltung.

In [ ]:
response = OLLAMA_CLIENT.chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Was ist der Unterschied zwischen KI und Machine Learning?"}
    ],
    think=False,
    options=build_options()
)
msg = response.get("message", {}) if isinstance(response, dict) else getattr(response, "message", {})
print(msg.get("content", "") if isinstance(msg, dict) else str(msg))

### 3.1 – Mehrrundigem Gespräch führen

In [ ]:
conversation = [
    {"role": "system",  "content": "Du bist ein präziser NLP-Tutor. Antworte auf Deutsch, maximal 3 Sätze."},
    {"role": "user",    "content": "Was ist ein Large Language Model?"},
]

resp = OLLAMA_CLIENT.chat(model=MODEL, messages=conversation, think=False, options=build_options())
first_msg = resp.get("message", {}) if isinstance(resp, dict) else getattr(resp, "message", {})
assistant_reply = first_msg.get("content", "") if isinstance(first_msg, dict) else str(first_msg)
conversation.append({"role": "assistant", "content": assistant_reply})

print("Assistent:", assistant_reply)

follow_up = "Wie viele Parameter hat GPT-4 ungefähr?"
conversation.append({"role": "user", "content": follow_up})

resp2 = OLLAMA_CLIENT.chat(model=MODEL, messages=conversation, think=False, options=build_options())
second_msg = resp2.get("message", {}) if isinstance(resp2, dict) else getattr(resp2, "message", {})
print("\nFollowup-Antwort:", second_msg.get("content", "") if isinstance(second_msg, dict) else str(second_msg))

## Teil 4 – Parameter erkunden ⏱️ ~25 min

### 4.1 – Temperature-Effekt

`temperature` kontrolliert die Zufälligkeit der Ausgabe:  
- `0.0` → meist sehr stabil / oft gleich, aber nicht in jeder Umgebung strikt identisch  
- `1.0` → kreativer / variabler  
- `> 1.5` → deutlich unvorhersehbarer

In [ ]:
prompt = "Erfinde einen kreativen Namen für ein KI-Startup."

print("Gleicher Prompt, verschiedene Temperaturen:\n")
for temp in [0.0, 0.5, 1.0, 1.5]:
    resp = OLLAMA_CLIENT.generate(model=MODEL, prompt=prompt, think=False, options=build_options(temperature=temp, seed=42))
    print(f"  T={temp:.1f} → {resp['response'].strip()}")

### 4.2 – System-Prompt Einfluss demonstrieren

In [ ]:
user_question = "Erkläre mir Backpropagation."

personas = {
    "Grundschullehrer":
        "Du erklärst komplexe Themen so einfach, dass sie ein 10-jähriges Kind verstehen kann.",
    "Wissenschaftler":
        "Du gibst exakte, mathematisch präzise Antworten mit Fachbegriffen.",
    "Comedian":
        "Du erklärst alles mit Witzen und Analogien zum Alltag.",
}

for name, system_prompt in personas.items():
    resp = OLLAMA_CLIENT.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_question},
        ],
        think=False,
        options=build_options(temperature=0.7)
    )
    msg = resp.get("message", {}) if isinstance(resp, dict) else getattr(resp, "message", {})
    text = (msg.get("content", "") if isinstance(msg, dict) else str(msg)).strip().replace("\n", " ")
    print(f"[{name}]\n{text[:200]}...\n")

### 4.3 – Latenz-Benchmark ⏱️ ~10 min

In [ ]:
import time

test_prompts = [
    "Nenne die Hauptstadt von Deutschland.",
    "Erkläre kurz, was ein neuronales Netz ist.",
]

print(f"{'Prompt (gekürzt)':<50} {'Tokens':>7} {'Sek':>6} {'Tok/s':>7}")
print("─" * 75)
for p in test_prompts:
    t0 = time.time()
    r = OLLAMA_CLIENT.generate(model=MODEL, prompt=p, options=build_options(temperature=0.0, num_predict=100))
    elapsed = max(time.time() - t0, 1e-6)
    tokens = r.get("eval_count", 0)
    if tokens == 0: tokens = r.get("prompt_eval_count", 0)
    print(f"{p[:48]:<50} {tokens:>7} {elapsed:>6.2f} {tokens/elapsed:>7.1f}")